# 1. Comprendre les données du projet ICU Nantes

Ce notebook explique, étape par étape et avec des données **synthétiques** (pas besoin de télécharger Landsat/Sentinel pour comprendre), comment fonctionne la partie "données" du projet :

1. Le paradoxe de résolution (100m vs 10m)
2. Le calcul des indices biophysiques (NDVI, NDWI, NDBI)
3. La structure du `Dataset` PyTorch qui alimente le modèle

L'objectif est pédagogique : on génère des données fictives qui *ressemblent* à de vraies images satellite pour visualiser ce que fait chaque étape, sans dépendance à Google Earth Engine.

## 1.1 Le paradoxe de résolution

- **Cible (Y)** : température de surface (LST), résolution brute **100m/pixel** → trop grossier pour voir une rue ou un alignement d'arbres.
- **Prédicteurs (X)** : imagerie optique + données urbaines, résolution **10m ou mieux**.

L'idée : utiliser le détail du X (10m) pour "affiner" la prédiction du Y (qui n'existe en vérité qu'à 100m), un peu comme si on demandait au modèle de dessiner les détails d'une photo floue en s'aidant d'une carte très précise du terrain.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

H, W = 256, 256

# On simule un patch urbain fictif : un centre-ville dense (chaud) entouré de végétation (frais)
xx, yy = np.meshgrid(np.linspace(-1, 1, W), np.linspace(-1, 1, H))
distance_centre = np.sqrt(xx**2 + yy**2)

# Bâti dense au centre, décroît vers les bords
densite_bati = np.clip(1 - distance_centre, 0, 1) ** 1.5

# Végétation : inverse du bâti + un peu de bruit (parcs dispersés)
vegetation = np.clip(1 - densite_bati + 0.3 * np.random.rand(H, W), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(densite_bati, cmap="inferno")
axes[0].set_title("Densité du bâti (simulée)")
axes[1].imshow(vegetation, cmap="Greens")
axes[1].set_title("Végétation (simulée)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 1.2 Construire les bandes spectrales et les indices

En vrai, Sentinel-2 fournit des bandes séparées (Bleu, Vert, Rouge, Proche Infrarouge, SWIR...).
Ici on les **fabrique** à partir de nos deux cartes (bâti / végétation) pour que les indices NDVI/NDWI/NDBI aient un sens visuel cohérent.

In [ ]:
# Bandes simulées (valeurs de réflectance, entre 0 et 1)
red   = 0.5 * densite_bati + 0.1 * (1 - vegetation) + 0.02 * np.random.rand(H, W)
nir   = 0.8 * vegetation + 0.1 * (1 - densite_bati) + 0.02 * np.random.rand(H, W)
green = 0.4 * vegetation + 0.2 * densite_bati + 0.02 * np.random.rand(H, W)
swir  = 0.6 * densite_bati + 0.1 * vegetation + 0.02 * np.random.rand(H, W)
blue  = 0.3 * (1 - vegetation) + 0.02 * np.random.rand(H, W)

def ndvi(nir, red):
    return (nir - red) / (nir + red + 1e-6)

def ndwi(green, nir):
    return (green - nir) / (green + nir + 1e-6)

def ndbi(swir, nir):
    return (swir - nir) / (swir + nir + 1e-6)

NDVI = ndvi(nir, red)
NDWI = ndwi(green, nir)
NDBI = ndbi(swir, nir)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
im0 = axes[0].imshow(NDVI, cmap="RdYlGn", vmin=-1, vmax=1)
axes[0].set_title("NDVI (végétation)")
im1 = axes[1].imshow(NDWI, cmap="Blues", vmin=-1, vmax=1)
axes[1].set_title("NDWI (humidité/eau)")
im2 = axes[2].imshow(NDBI, cmap="OrRd", vmin=-1, vmax=1)
axes[2].set_title("NDBI (bâti/imperméabilisation)")
for ax, im in zip(axes, [im0, im1, im2]):
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print("NDVI : proche de 1 = végétation dense, proche de -1 = surface minérale/eau")
print("NDBI : proche de 1 = bâti/surface imperméabilisée dense")

## 1.3 Simuler la cible (LST) à 100m et le "downsampling"

On simule une température de surface (LST) cohérente avec notre bâti (plus il y a de bâti, plus il fait chaud = effet îlot de chaleur), puis on la dégrade à 100m pour reproduire la contrainte réelle de Landsat.

In [ ]:
import torch
import torch.nn.functional as F

# Température de base 22°C + effet du bâti (jusqu'à +8°C) - effet rafraîchissant de la végétation (jusqu'à -3°C)
LST_hd = 22 + 8 * densite_bati - 3 * vegetation + 0.3 * np.random.randn(H, W)

lst_tensor = torch.tensor(LST_hd, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

# Simule la dégradation à 100m (facteur 10 car nos pixels "10m" -> pooling 10x10 = 100m)
lst_100m = F.avg_pool2d(lst_tensor, kernel_size=10, stride=10)

# On la "remonte" à la même taille juste pour visualiser la perte de détail (pas pour l'entraînement)
lst_100m_visu = F.interpolate(lst_100m, size=(H, W), mode="nearest")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(LST_hd, cmap="inferno")
axes[0].set_title("LST 'vérité' à 10m (ce qu'on veut retrouver)")
im1 = axes[1].imshow(lst_100m_visu[0, 0], cmap="inferno")
axes[1].set_title("LST dégradée à 100m (ce que Landsat voit vraiment)")
for ax, im in zip(axes, [im0, im1]):
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print("C'est exactement ce paradoxe (droite = vérité terrain dispo, gauche = ce qu'on veut reconstruire)")
print("que le modèle de super-résolution guidée essaie de résoudre.")

## 1.4 Le `Dataset` PyTorch (version pédagogique)

Ci-dessous, une version simplifiée de `NantesICUDataset` qui fonctionne directement sur des tableaux numpy en mémoire (au lieu de fichiers raster sur disque), pour que tu puisses voir immédiatement ce que `__getitem__` renvoie.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class NantesICUDatasetDemo(Dataset):
    """Version pédagogique : les patchs sont déjà en mémoire (numpy), pas lus depuis des .tif."""

    def __init__(self, X_list, Y_list):
        # X_list : liste de tableaux (C, H, W)
        # Y_list : liste de tableaux (H, W)
        self.X_list = X_list
        self.Y_list = Y_list

    def __len__(self):
        return len(self.X_list)

    def __getitem__(self, idx):
        x = torch.tensor(self.X_list[idx], dtype=torch.float32)
        y = torch.tensor(self.Y_list[idx], dtype=torch.float32).unsqueeze(0)
        x = torch.nan_to_num(x, nan=0.0)
        y = torch.nan_to_num(y, nan=0.0)
        return x, y

# On empile nos 7 canaux (RGB + NDVI + NDWI + canopée(≈vegetation) + bâti)
X_patch = np.stack([red, green, blue, NDVI, NDWI, vegetation, densite_bati], axis=0)  # (7, H, W)
Y_patch = LST_hd  # (H, W)

dataset_demo = NantesICUDatasetDemo(X_list=[X_patch], Y_list=[Y_patch])
loader_demo = DataLoader(dataset_demo, batch_size=1)

x_batch, y_batch = next(iter(loader_demo))
print("Forme de X (entrée modèle) :", x_batch.shape)   # (Batch, C=7, H, W)
print("Forme de Y (cible) :", y_batch.shape)            # (Batch, 1, H, W)

## À retenir

- Le modèle reçoit un tenseur `(Batch, 7, H, W)` : 3 bandes couleur + NDVI + NDWI + canopée + bâti.
- La cible `(Batch, 1, H, W)` est la température à haute résolution, mais **on ne dispose vraiment que de sa version dégradée à 100m** — d'où la fonction de perte de cohérence qu'on verra dans le notebook 2.
- Passe au notebook `02_comprendre_le_modele_UNet.ipynb` pour voir comment le U-Net exploite ces données.